# Smoke Test — End-to-End Validation
10 FEVER examples, real API calls, no plots. Goal: verify all modules are wired correctly before running full experiments.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
from pathlib import Path

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

print("Config loaded:", list(cfg.keys()))

In [ ]:
from src.data.fever_loader import load_fever

examples = load_fever(
    "../" + cfg["dataset"]["fever_dev"],
    max_examples=10,
)

print(f"Loaded {len(examples)} examples")
print("Keys:", list(examples[0].keys()))
print("Labels:", [e["label"] for e in examples])

In [ ]:
from src.data.poisoner import poison_dataset

poisoned = poison_dataset(examples, poison_rate=0.5, seed=cfg["seed"])
print(f"Poisoned {len(poisoned)} examples at rate 0.5")

In [ ]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever

emb_cache = os.path.join("../" + cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])

print(f"Embedder: {cfg["retrieval"]["embedding_model"]}, k={cfg["retrieval"]["k"]}")

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

from src.generation.llm_client import OpenAIClient

llm_cache = os.path.join("../" + cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
llm = OpenAIClient(
    model="gpt-4o-mini",
    temperature=cfg["models"]["temperature"],
    cache_dir=llm_cache,
)
print("LLM: gpt-4o-mini")

In [ ]:
from src.evaluation.scorer import run as run_scorer

with embedder, llm:
    metrics = run_scorer(
        examples=examples,
        retriever=retriever,
        llm=llm,
        prompt_type="standard",
        distractor_pool_size=cfg["retrieval"]["distractor_pool_size"],
        seed=cfg["seed"],
        self_consistency_runs=1,
    )

print("\n=== Smoke Test Results (poison_rate=0.0) ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Quick sanity check: same run with poison_rate=0.5
with embedder, llm:
    metrics_poisoned = run_scorer(
        examples=poisoned,
        retriever=retriever,
        llm=llm,
        prompt_type="standard",
        distractor_pool_size=cfg["retrieval"]["distractor_pool_size"],
        seed=cfg["seed"],
        self_consistency_runs=1,
    )

print("\n=== Smoke Test Results (poison_rate=0.5) ===")
for k, v in metrics_poisoned.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n✓ Pipeline wired correctly — proceed to notebook 01." if metrics_poisoned else "✗ Check errors above.")